# Module 2 - Partie 4 : Découpage et pipeline scikit-learn

## Objectif

Construire un premier protocole de modélisation reproductible pour détecter les transactions frauduleuses du dataset PaySim.

Cette partie couvre le découpage chronologique, la sélection de six variables disponibles au moment de la transaction, le prétraitement dans des pipelines scikit-learn, un modèle naïf de référence et une régression logistique. Les modèles sont ajustés sur l'entraînement et comparés sur la validation ; le jeu de test reste à l'écart.

La partie 5 approfondira l'évaluation et la sélection du modèle.


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


In [2]:
fichier_parquet = Path("../paysim_echantillon.parquet")

print("Fichier trouvé :", fichier_parquet.exists())

df = pd.read_parquet(fichier_parquet)

print("Dimensions :", df.shape)
display(df.head())


Fichier trouvé : True
Dimensions : (555108, 10)


,tx_id,step,heure,type,amount,oldbalanceOrg,oldbalanceDest,nb_tx_orig_avant,nb_recu_dest_avant,isFraud
0,48,1,0,CASH_OUT,56953.90,1942.02,70253.0,0,0,0
1,59,1,0,TRANSFER,62610.80,79114.00,517.0,0,0,0
2,73,1,0,CASH_OUT,94253.33,25203.05,99773.0,0,0,0
3,84,1,0,TRANSFER,125872.53,0.00,348512.0,0,0,0
4,86,1,0,TRANSFER,1505626.01,0.00,29031.0,0,0,0


## 1. Découpage chronologique : méthode et justification

Un découpage chronologique est préférable à un découpage aléatoire ici : il respecte l'ordre dans lequel les transactions seraient observées en situation réelle. Un découpage aléatoire mêlerait passé et futur dans chaque ensemble et pourrait donner une estimation trop optimiste en masquant l'évolution des comportements frauduleux.

Les frontières sont déterminées par les quantiles 60 % et 80 % de la colonne **step**. Comme le quantile est calculé sur une ligne par transaction, il est implicitement pondéré par le nombre de transactions de chaque step ; il ne divise donc pas la durée simulée en trois parties égales.

Pour cet échantillon, les frontières obtenues sont : entraînement, steps 1 à 281 ; validation, steps 282 à 354 ; test, steps 355 à 743. Les effectifs et les taux de fraude correspondants seront vérifiés ci-dessous.

Les taux de fraude observés diffèrent fortement entre périodes. Cette évolution constitue un changement de distribution : les résultats de validation doivent être interprétés dans ce contexte, et le test futur doit rester réservé à l'évaluation finale.

In [3]:
print("Période couverte (step) :", df["step"].min(), "à", df["step"].max())

print()
print("Répartition de la variable cible :")
print(df["isFraud"].value_counts())

print()
print("Taux de fraude (%) :")
print((df["isFraud"].value_counts(normalize=True) * 100).round(4))

print()
print("Valeurs manquantes par colonne :")
print(df.isna().sum())

Période couverte (step) : 1 à 743

Répartition de la variable cible :
isFraud
0    553512
1      1596
Name: count, dtype: int64

Taux de fraude (%) :
isFraud
0    99.7125
1     0.2875
Name: proportion, dtype: float64

Valeurs manquantes par colonne :
tx_id                 0
step                  0
heure                 0
type                  0
amount                0
oldbalanceOrg         0
oldbalanceDest        0
nb_tx_orig_avant      0
nb_recu_dest_avant    0
isFraud               0
dtype: int64


In [4]:
# Les quantiles de step sont calculés sur toutes les transactions.
# Chaque transaction contribue donc une observation au calcul du quantile.
borne_train = int(df["step"].quantile(0.60))
borne_validation = int(df["step"].quantile(0.80))

print("Fin entraînement :", borne_train)
print("Fin validation :", borne_validation)
print("Début test :", borne_validation + 1)

# Affectation des périodes pour contrôler les effectifs et les taux de fraude.
df_verification = df.copy()
df_verification["periode"] = pd.Series(index=df_verification.index, dtype="object")
df_verification.loc[df_verification["step"] <= borne_train, "periode"] = "Entrainement"
df_verification.loc[
    (df_verification["step"] > borne_train)
    & (df_verification["step"] <= borne_validation),
    "periode",
] = "Validation"
df_verification.loc[df_verification["step"] > borne_validation, "periode"] = "Test"


Fin entraînement : 281
Fin validation : 354
Début test : 355


In [5]:
resume = df_verification.groupby("periode").agg(
    transactions=("isFraud", "size"),
    fraudes=("isFraud", "sum"),
)

resume["part_transactions_pct"] = (
    100 * resume["transactions"] / len(df)
).round(2)
resume["taux_fraude_pct"] = (
    100 * resume["fraudes"] / resume["transactions"]
).round(4)

print(resume)


              transactions  fraudes  part_transactions_pct  taux_fraude_pct
periode                                                                    
Entrainement        335288      617                  60.40           0.1840
Test                110516      843                  19.91           0.7628
Validation          109304      136                  19.69           0.1244


In [6]:
# Création des trois ensembles chronologiques
df_train = df[df["step"] <= borne_train].copy()
df_val = df[
    (df["step"] > borne_train)
    & (df["step"] <= borne_validation)
].copy()
df_test = df[df["step"] > borne_validation].copy()

print("Entraînement :", df_train.shape)
print("Validation   :", df_val.shape)
print("Test         :", df_test.shape)

# Vérification de l'ordre, de l'absence de chevauchement et de l'exhaustivité.
assert df_train["step"].max() < df_val["step"].min()
assert df_val["step"].max() < df_test["step"].min()
assert len(df_train) + len(df_val) + len(df_test) == len(df)
print("Découpage chronologique vérifié.")


Entraînement : (335288, 10)
Validation   : (109304, 10)
Test         : (110516, 10)
Découpage chronologique vérifié.


## 2. Variables prédictives et prévention des fuites

La cible est **isFraud** (0 : transaction légitime ; 1 : fraude). Les six variables prédictives conservées sont **type**, **amount**, **oldbalanceOrg**, **oldbalanceDest**, **nb_tx_orig_avant** et **nb_recu_dest_avant**.

**step** sert exclusivement à définir les périodes et n'entre pas dans les prédicteurs. **tx_id** est un identifiant technique et **heure**, dérivée de step, est exclue du modèle initial. Les variables post-transaction **newbalanceOrig** et **newbalanceDest**, ainsi que **isFlaggedFraud**, ne sont pas utilisées.

In [7]:
# Variables explicatives retenues
variables = [
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "nb_tx_orig_avant",
    "nb_recu_dest_avant"
]

cible = "isFraud"

# Séparation des variables explicatives et de la cible
X_train = df_train[variables].copy()
y_train = df_train[cible].copy()

X_val = df_val[variables].copy()
y_val = df_val[cible].copy()

X_test = df_test[variables].copy()
y_test = df_test[cible].copy()

print("Variables retenues :", X_train.columns.tolist())
print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("X_val   :", X_val.shape)
print("X_test  :", X_test.shape)

Variables retenues : ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'nb_tx_orig_avant', 'nb_recu_dest_avant']
X_train : (335288, 6)
y_train : (335288,)
X_val   : (109304, 6)
X_test  : (110516, 6)


## 3. Prétraitement scikit-learn

Un **ColumnTransformer** applique deux branches : imputation médiane puis **StandardScaler** aux cinq variables numériques ; imputation par la modalité majoritaire puis **OneHotEncoder** à **type**.

Le préprocesseur ci-dessous reste un modèle de configuration non ajusté. Chaque pipeline reçoit sa propre copie indépendante. Avec **Pipeline.fit(X_train, y_train)**, les statistiques d'imputation, l'échelle et les catégories sont apprises uniquement sur l'entraînement ; la validation est transformée avec ces paramètres appris, jamais réajustés sur elle.

In [8]:
variables_numeriques = [
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "nb_tx_orig_avant",
    "nb_recu_dest_avant",
]
variables_categorielles = ["type"]

pipeline_numerique = Pipeline(steps=[
    ("imputation", SimpleImputer(strategy="median")),
    ("standardisation", StandardScaler()),
])
pipeline_categoriel = Pipeline(steps=[
    ("imputation", SimpleImputer(strategy="most_frequent")),
    ("encodage", OneHotEncoder(handle_unknown="ignore")),
])

preprocesseur = ColumnTransformer(transformers=[
    ("numerique", pipeline_numerique, variables_numeriques),
    ("categoriel", pipeline_categoriel, variables_categorielles),
])
print("Préprocesseur non ajusté créé.")


Préprocesseur non ajusté créé.


## 4. Modèle naïf de référence

Un modèle naïf (DummyClassifier) est utilisé comme référence avant l'entraînement de la régression logistique.

La stratégie "prior" prédit systématiquement la classe majoritaire et fournit des probabilités correspondant aux fréquences observées dans l'ensemble d'entraînement.

Ce modèle permet de montrer pourquoi une forte accuracy ne suffit pas dans un problème de détection de fraude fortement déséquilibré.

In [9]:
# Copie indépendante du préprocesseur pour ce pipeline.
pipeline_naif = Pipeline(steps=[
    ("pretraitement", clone(preprocesseur)),
    ("modele", DummyClassifier(strategy="prior")),
])

# Ajustement sur les seules données d'entraînement.
pipeline_naif.fit(X_train, y_train)

print("Modèle naïf entraîné sur l'entraînement.")
print("Classes :", pipeline_naif.named_steps["modele"].classes_)
print("Fréquences des classes :", pipeline_naif.named_steps["modele"].class_prior_)


Modèle naïf entraîné sur l'entraînement.
Classes : [0 1]
Fréquences des classes : [0.99815979 0.00184021]


## 5. Régression logistique pondérée

La régression logistique est le premier modèle supervisé. Le poids **class_weight="balanced"** compense le déséquilibre des classes à partir des seules fréquences d'entraînement. Son pipeline possède un préprocesseur indépendant de celui du modèle naïf et est ajusté exclusivement sur l'entraînement.

In [10]:
pipeline_logistique = Pipeline(steps=[
    ("pretraitement", clone(preprocesseur)),
    ("modele", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

pipeline_logistique.fit(X_train, y_train)

print("Régression logistique entraînée sur l'entraînement.")
print("Nombre de variables après prétraitement :",
      pipeline_logistique.named_steps["modele"].n_features_in_)


Régression logistique entraînée sur l'entraînement.
Nombre de variables après prétraitement : 7


## 6. Comparaison des modèles sur la validation

Les prédictions et les métriques sont calculées uniquement sur la validation. Le seuil de décision par défaut de 0,5 est conservé pour les deux modèles ; il n'est pas optimisé à ce stade. Le tableau est calculé à partir des prédictions lors de l'exécution du notebook.


In [11]:
# Prédictions sur la validation uniquement ; aucun appel au jeu de test.
pred_naif = pipeline_naif.predict(X_val)
pred_logistique = pipeline_logistique.predict(X_val)

print("Transactions en validation :", len(X_val))
print("Transactions signalées par le modèle naïf :", int((pred_naif == 1).sum()))
print("Transactions signalées par la régression logistique :",
      int((pred_logistique == 1).sum()))


Transactions en validation : 109304
Transactions signalées par le modèle naïf : 0
Transactions signalées par la régression logistique : 8718


In [13]:
# Les métriques sont recalculées depuis les prédictions, sans valeurs codées en dur.
tableau_comparaison = pd.DataFrame([
    {
        "Modèle": "Modèle naïf",
        "Transactions signalées": int((pred_naif == 1).sum()),
        "Précision": precision_score(y_val, pred_naif, zero_division=0),
        "Rappel": recall_score(y_val, pred_naif, zero_division=0),
        "F1-score": f1_score(y_val, pred_naif, zero_division=0),
    },
    {
        "Modèle": "Régression logistique",
        "Transactions signalées": int((pred_logistique == 1).sum()),
        "Précision": precision_score(y_val, pred_logistique, zero_division=0),
        "Rappel": recall_score(y_val, pred_logistique, zero_division=0),
        "F1-score": f1_score(y_val, pred_logistique, zero_division=0),
    },
]).set_index("Modèle")

# Affichage du tableau sans dépendance supplémentaire
tableau_affichage = tableau_comparaison.copy()

for colonne in ["Précision", "Rappel", "F1-score"]:
    tableau_affichage[colonne] = tableau_affichage[colonne].map(
        lambda valeur: f"{valeur:.2%}"
    )

display(tableau_affichage)


,Transactions signalées,Précision,Rappel,F1-score
Modèle,,,,
Modèle naïf,0,0.00%,0.00%,0.00%
Régression logistique,8718,1.31%,83.82%,2.58%


## 7. Conclusion et périmètre de la partie 5

Cette partie a établi un protocole chronologique en trois périodes, retenu six variables explicatives sans inclure les variables exclues pour prévenir les fuites, et construit des prétraitements indépendants au sein de pipelines scikit-learn. Le modèle naïf et la régression logistique pondérée ont été entraînés uniquement sur l'entraînement, puis comparés sur la validation à l'aide du nombre de transactions signalées, de la précision, du rappel et du F1-score. Le tableau présente les résultats recalculés à l'exécution avec le seuil par défaut de 0,5, qui n'a pas été optimisé.

Les performances de validation sont à interpréter en tenant compte des différences de taux de fraude entre périodes. Elles constituent une comparaison préliminaire, et non une évaluation finale de généralisation. Le jeu de test n'a été utilisé ni pour ajuster les modèles ni pour calculer les résultats présentés.

La partie 5 pourra traiter séparément la validation croisée temporelle, l'évaluation finale sur le test, des métriques approfondies adaptées au déséquilibre, des intervalles de confiance et la justification du seuil de décision.
